<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/6_Competencia_Seleccion_Mejor_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# 🏆 **Tercer parcial: Selección del Mejor Modelo**

**ESTÁ PROHIBIDO EL USO DE GRANDES MODELOS DE LENGUAJE COMO CHATGPT, CLAUDE, GEMINI, ENTRE OTROS, PARA RESOLVER ESTE EJERCICIO**

**Trabajo en grupos de 3**

**Objetivo:** Predecir las ventas de una compañía (`Sales`) teniendo en cuenta su inversión en publicidad.

**Dataset:** `train_df_ventas.csv` disponible en el repositorio del curso.

**IMPORTANTE: Los datos cargados solo corresponden a `train`.**

**Metodología:**
1.  Cargar y explorar los datos.
2.  Preprocesar los datos si es necesario.
3.  De los siguientes modelos, entrenar por lo menos 2:
    - Regresión Lineal
    - Regresión Polinómica
    - KNN Regressor
    - Decision Tree Regressor
    - Random Forest Regressor
    - Gradient Boosting Regressor
    - XGBoost Regressor
4.  Si lo considera necesario, usar `GridSearchCV` con Validación Cruzada (`cv=5`) para optimizar los hiperparámetros. La métrica de optimización debe ser el **RMSE** (Root Mean Squared Error), por lo que debe usar `scoring='neg_root_mean_squared_error'` (el valor será negativo y se multiplicará por -1 al final).
5.  Comparar los modelos y seleccionar el mejor, teniendo en cuenta el menor RMSE.

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Tercer_Parcial_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/Bsy2U83tbc. No olvide indicar claramente cuál es el modelo seleccionado.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Calificación**

La docente, evaluará el modelo seleccionado por ustedes en el `test set`.

El proceso seguirá estas reglas:

- **Criterio de Ganador:** El equipo que tenga todo el procedimiento correcto y obtenga el Root Mean Squared Error (RMSE) más bajo en el test set recibirá una calificación de 5.0.

- **Criterio de Desempate:** En caso de empate en el RMSE, se otorgará la ventaja al equipo que haya entrenado y evaluado más modelos.

- **Escalafón de Notas:** A partir del primer puesto, se restará 0.1 a la nota final por cada posición inferior (2º lugar: 4.9, 3er lugar: 4.8, etc.).

- **Validación de Procedimiento:** Es obligatorio que el código sea reproducible por la docente (no olivde colocar las semillas en los procesos aleatorios). Si el script contiene errores, el equipo quedará fuera de la competencia y se dará una calificación acorde a lo que esté correcto.

**Explicación de las variables:**

- Sales: Ventas (millones USD). --> **Esta es la variable objetivo**
- TV: Gasto en promoción televisiva (millones USD).
- Radio: Gasto en promoción radiofónica (millones USD).
- Social Media: Gasto en promoción en redes sociales (millones USD).
- Influencer: Indica si la promoción se realizó en colaboración con Mega, Macro, Nano o Micro influencers.


**Nombres estudiantes del equipo:**

- Andres Santiago Cristiano Trujillo
- Samuel David Huertas Infante
- Juan Diego Villabón López

# **Desarrollo**

# **Indica claramente cuál modelo seleccionaste como el mejor**

#### *Cargar bases de datos y liberias*

In [49]:
import os
import warnings
warnings.filterwarnings('ignore')

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBClassifier

# Métricas y Tuning
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from multiprocessing import ProcessError
from sklearn.metrics import mean_squared_error, r2_score

In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [51]:
path = '/content/drive/MyDrive/2026-i-curso-ia-para-economia/datasets/Parcial 3'

In [52]:
os.chdir(path)

In [53]:
df = pd.read_csv('train_df_ventas.csv')
df.head()

,TV,Social Media,Influencer,Radio,Sales
0,17.579874,1.218985,Macro,22.282240,173.612079
1,14.588548,5.478278,Micro,22.097499,140.540703
2,25.183695,2.279885,Mega,22.197715,197.352805
3,12.898275,1.831455,Nano,21.741263,184.047821
4,28.380520,4.338470,Nano,22.607219,318.627485


#### *Preprocesamiento*

In [54]:
X = df.drop('Sales', axis=1)
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print("Tamaño de X_Train original:", X_train.shape)
print("Tamaño de X_Test original:", X_test.shape)


numerical_features = X_train.select_dtypes(include=np.number).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', 'passthrough', numerical_features)
    ],
    remainder='passthrough'
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nForma de X_Train procesado:", X_train_processed.shape)
print("Forma de X_Test procesado:", X_test_processed.shape)

Tamaño de X_Train original: (2908, 4)
Tamaño de X_Test original: (728, 4)

Forma de X_Train procesado: (2908, 6)
Forma de X_Test procesado: (728, 6)


### **1. Gradient Boosting**

In [59]:
gb_model = GradientBoostingRegressor(random_state=42)

gb_model.fit(X_train_processed, y_train)

y_pred_gb = gb_model.predict(X_test_processed)

gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_r2 = r2_score(y_test, y_pred_gb)

print(f"RMSE: {gb_rmse:.4f}")
print(f"R²: {gb_r2:.4f}")

RMSE: 41.5562
R²: 0.7906


### **2. Random Forest Regressor**

In [56]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

rf_model = RandomForestRegressor(n_estimators=500,max_depth=10,min_samples_leaf=2,random_state=42,n_jobs=-1)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [61]:
print("Iniciando GridSearchCV para Random Forest")

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_leaf': [1, 2],
}

rf_base_model = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search_rf = GridSearchCV(
    estimator=rf_base_model,
    param_grid=param_grid_rf,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)


grid_search_rf.fit(X_train_processed, y_train)

print("¡Búsqueda completada!")
print(f"Mejores parámetros encontrados: {grid_search_rf.best_params_}")

best_rf_model = grid_search_rf.best_estimator_
y_pred_rf_optimizado = best_rf_model.predict(X_test_processed)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf_optimizado))
rf_r2 = r2_score(y_test, y_pred_rf_optimizado)

print(f"RMSE optimizado: {rf_rmse:.4f}")
print(f"R² optimizado: {rf_r2:.4f}")

Iniciando GridSearchCV para Random Forest
Fitting 5 folds for each of 8 candidates, totalling 40 fits
¡Búsqueda completada!
Mejores parámetros encontrados: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 200}
RMSE optimizado: 42.4324
R² optimizado: 0.7816


El mejor modelo para ser usado es el Gradient boosting ya que, en comparacion random  